# Add CNV to Merged Datasets

Adds CNV MB-selected features to each of the 8 existing Clinical+RNA merged files.

**Run cells in order.**

**Input:** `MERGE/01_ultra_conservative.csv` ... `08_composite.csv`  
**Output:** same files overwritten with added `CNV_` columns

## Setup — Imports, Paths, Best CNV Config per Dataset

In [3]:
"""
ADD CNV: Add CNV features to existing Clinical+RNA merged datasets.

For each of 8 merged files in MERGE/:
  1. Load existing merged file (Clinical + RNA)
  2. Read CNV summary_all_results.csv -> pick best C-index config
  3. Load CNV gene list from mb_results/{dataset}/{algo}_alpha{alpha}_genes.txt
  4. Load CNV statistical_filtered CSV, keep only selected genes
  5. Normalize CNV IDs, filter to Primary Tumor, average duplicates
  6. Add CNV_ prefix, merge with existing dataset
  7. Overwrite file in MERGE/ (same name, more columns)

Script location: 01_Causal_feature_extraction/
Input/Output:    01_Causal_feature_extraction/MERGE/
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------------
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    cwd = Path.cwd()
    SCRIPT_DIR = cwd.parent if cwd.name == "MERGE" else cwd

CNV_DIR   = SCRIPT_DIR / "CNV"
FILT_DIR  = CNV_DIR / "statistical_filtered"
MB_DIR    = CNV_DIR / "mb_results"
MERGE_DIR = SCRIPT_DIR / "MERGE"

print(f"Script dir:  {SCRIPT_DIR}")
print(f"CNV MB:      {MB_DIR.exists()}")
print(f"MERGE dir:   {MERGE_DIR.exists()}")

# ---------------------------------------------------------------------------
# DATASET PAIRING: merged file -> CNV dataset (matched by number 1-8)
# ---------------------------------------------------------------------------
DATASET_PAIRS = {
    "01_ultra_conservative": "cnv_1_ultra_conservative_768genes",
    "02_conservative":       "cnv_2_conservative_1128genes",
    "03_standard":           "cnv_3_standard_1813genes",
    "04_fdr_significant":    "cnv_4_fdr_significant_1000genes",
    "05_balanced":           "cnv_5_balanced_1130genes",
    "06_correlation":        "cnv_6_correlation_1127genes",
    "07_top_correlated":     "cnv_7_top_correlated_200genes",
    "08_composite":          "cnv_8_composite_1000genes",
}

# STEP 1: LOAD CNV SUMMARY AND SELECT BEST CONFIG PER DATASET
print("\n" + "="*70)
print("STEP 1: SELECT BEST CNV CONFIG PER DATASET (highest C-index)")
print("="*70)

summary = pd.read_csv(MB_DIR / "summary_all_results.csv")

best_cnv = (
    summary
    .sort_values("c_index", ascending=False)
    .groupby("dataset", sort=False)
    .first()
    .reset_index()
)[["dataset", "algorithm", "alpha", "c_index", "n_causal_genes"]]

print(best_cnv.to_string(index=False))

Script dir:  C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction
CNV MB:      True
MERGE dir:   True

STEP 1: SELECT BEST CNV CONFIG PER DATASET (highest C-index)
                          dataset algorithm  alpha  c_index  n_causal_genes
         cnv_5_balanced_1130genes      IAMB   0.20 0.520790              50
    cnv_7_top_correlated_200genes      IAMB   0.10 0.519879              50
      cnv_6_correlation_1127genes      GSMB   0.05 0.505270              50
        cnv_8_composite_1000genes      GSMB   0.10 0.504198              50
  cnv_4_fdr_significant_1000genes      GSMB   0.10 0.504198              50
         cnv_3_standard_1813genes      IAMB   0.20 0.498541              50
     cnv_2_conservative_1128genes      GSMB   0.20 0.495351              50
cnv_1_ultra_conservative_768genes      GSMB   0.10 0.492490              50


## Helper Functions

In [5]:
# STEP 2: HELPERS
def normalize_id(sample_id):
    """TCGA-D8-A146-01A  ->  TCGA-D8-A146"""
    parts = str(sample_id).split("-")
    return "-".join(parts[:3]) if len(parts) >= 3 else str(sample_id)


def load_genes(mb_dir, dataset, algorithm, alpha):
    for fmt in [f"{alpha:.2f}", str(alpha)]:
        p = mb_dir / dataset / f"{algorithm}_alpha{fmt}_genes.txt"
        if p.exists():
            return [l.strip() for l in p.read_text().splitlines() if l.strip()]
    raise FileNotFoundError(
        f"Gene file not found: {mb_dir / dataset / f'{algorithm}_alpha{alpha:.2f}_genes.txt'}"
    )


def load_cnv(cnv_dataset, algorithm, alpha):
    """Load CNV data filtered to selected genes, normalized to patient IDs."""
    candidates = list(FILT_DIR.glob(f"{cnv_dataset}*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No CNV file found for '{cnv_dataset}'")
    cnv_file = candidates[0]

    genes     = load_genes(MB_DIR, cnv_dataset, algorithm, alpha)
    cnv       = pd.read_csv(cnv_file, index_col=0)
    available = [g for g in genes if g in cnv.columns]
    missing_g = [g for g in genes if g not in cnv.columns]
    if missing_g:
        print(f"  Warning: {len(missing_g)} CNV genes not in file")
    cnv = cnv[available].copy()
    print(f"  CNV loaded: {cnv.shape}  ({cnv_file.name})")

    # Primary Tumor only
    tumor_mask = cnv.index.str.contains(r"-01[A-Z]?$", regex=True)
    cnv = cnv[tumor_mask].copy()
    print(f"  After Primary Tumor filter: {len(cnv)} samples")

    # normalize IDs
    cnv.index = [normalize_id(i) for i in cnv.index]

    # average duplicates
    if cnv.index.duplicated().any():
        n_pts = cnv.index[cnv.index.duplicated(keep=False)].nunique()
        print(f"  Averaging {n_pts} patients with duplicate CNV samples")
        cnv = cnv.groupby(cnv.index).mean()

    # add CNV_ prefix
    cnv.columns = [f"CNV_{c}" for c in cnv.columns]
    return cnv

print("\nHelper functions ready")


Helper functions ready


## Add CNV to All 8 Datasets

In [7]:
# STEP 3: ADD CNV TO ALL 8 DATASETS
print("\n" + "="*70)
print("STEP 3: ADD CNV TO ALL 8 MERGED DATASETS")
print("="*70)

results = []

for short_name, cnv_dataset in DATASET_PAIRS.items():
    merged_file = MERGE_DIR / f"{short_name}.csv"
    if not merged_file.exists():
        print(f"\nSkipping {short_name} — file not found: {merged_file}")
        continue

    # get best CNV config for this dataset
    row = best_cnv[best_cnv["dataset"] == cnv_dataset]
    if row.empty:
        print(f"\nNo CNV summary entry for {cnv_dataset} — skipping")
        continue
    row = row.iloc[0]

    print(f"\n{'─'*70}")
    print(f"[{short_name}]")
    print(f"  CNV dataset: {cnv_dataset}")
    print(f"  Config:      {row['algorithm']}  alpha={row['alpha']}  C-index={row['c_index']:.4f}")

    try:
        # load existing merged (Clinical + RNA)
        merged = pd.read_csv(merged_file, index_col=0)
        print(f"  Existing:    {merged.shape}  "
              f"(CLIN_={sum(1 for c in merged.columns if c.startswith('CLIN_'))}, "
              f"RNA_={sum(1 for c in merged.columns if c.startswith('RNA_'))})")

        # load CNV
        cnv = load_cnv(cnv_dataset, row["algorithm"], float(row["alpha"]))

        # align and merge
        common = sorted(set(merged.index) & set(cnv.index))
        print(f"  Common patients: {len(common)}  "
              f"(merged={len(merged)}, CNV={len(cnv)})")
        if not common:
            raise ValueError("No common patients")

        final = pd.concat([merged.loc[common], cnv.loc[common]], axis=1)

        # verify
        assert final.isna().sum().sum() == 0, \
            f"Missing values: {final.isna().sum().sum()}"
        n_clin = sum(1 for c in final.columns if c.startswith("CLIN_"))
        n_rna  = sum(1 for c in final.columns if c.startswith("RNA_"))
        n_cnv  = sum(1 for c in final.columns if c.startswith("CNV_"))
        print(f"  Final shape: {final.shape}  "
              f"CLIN_={n_clin}  RNA_={n_rna}  CNV_={n_cnv}  | no missing")

        # overwrite same file
        final.to_csv(merged_file)
        print(f"  Saved: {merged_file.name}")

        results.append({
            "file":             short_name + ".csv",
            "n_samples":        final.shape[0],
            "n_clin_features":  n_clin,
            "n_rna_features":   n_rna,
            "n_cnv_features":   n_cnv,
            "n_total_features": final.shape[1],
            "cnv_dataset":      cnv_dataset,
            "cnv_algorithm":    row["algorithm"],
            "cnv_alpha":        row["alpha"],
            "cnv_c_index":      row["c_index"],
        })

    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback; traceback.print_exc()


STEP 3: ADD CNV TO ALL 8 MERGED DATASETS

──────────────────────────────────────────────────────────────────────
[01_ultra_conservative]
  CNV dataset: cnv_1_ultra_conservative_768genes
  Config:      GSMB  alpha=0.1  C-index=0.4925
  Existing:    (1071, 194)  (CLIN_=144, RNA_=50)
  CNV loaded: (1043, 50)  (cnv_1_ultra_conservative_768genes.csv)
  After Primary Tumor filter: 1041 samples
  Common patients: 1038  (merged=1071, CNV=1041)
  Final shape: (1038, 244)  CLIN_=144  RNA_=50  CNV_=50  | no missing
  Saved: 01_ultra_conservative.csv

──────────────────────────────────────────────────────────────────────
[02_conservative]
  CNV dataset: cnv_2_conservative_1128genes
  Config:      GSMB  alpha=0.2  C-index=0.4954
  Existing:    (1071, 194)  (CLIN_=144, RNA_=50)
  CNV loaded: (1043, 50)  (cnv_2_conservative_1128genes.csv)
  After Primary Tumor filter: 1041 samples
  Common patients: 1038  (merged=1071, CNV=1041)
  Final shape: (1038, 244)  CLIN_=144  RNA_=50  CNV_=50  | no missing
 

## Summary

In [9]:
# STEP 4: SUMMARY
print("\n" + "="*70)
print("STEP 4: SUMMARY")
print("="*70)

if results:
    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    df.to_csv(MERGE_DIR / "merge_summary_with_cnv.csv", index=False)

    print(f"\nClinical features same across all: "
          f"{df['n_clin_features'].nunique() == 1}  ({df['n_clin_features'].iloc[0]})")
    print(f"RNA features vary:  {df['n_rna_features'].nunique() > 1}  "
          f"(range {df['n_rna_features'].min()}–{df['n_rna_features'].max()})")
    print(f"CNV features vary:  {df['n_cnv_features'].nunique() > 1}  "
          f"(range {df['n_cnv_features'].min()}–{df['n_cnv_features'].max()})")
    print(f"Total features:     {df['n_total_features'].min()}–{df['n_total_features'].max()}")
    print(f"Samples:            {df['n_samples'].min()}–{df['n_samples'].max()}")
    print(f"\nSaved: {MERGE_DIR / 'merge_summary_with_cnv.csv'}")
else:
    print("No datasets processed successfully.")


STEP 4: SUMMARY
                     file  n_samples  n_clin_features  n_rna_features  n_cnv_features  n_total_features                       cnv_dataset cnv_algorithm  cnv_alpha  cnv_c_index
01_ultra_conservative.csv       1038              144              50              50               244 cnv_1_ultra_conservative_768genes          GSMB       0.10     0.492490
      02_conservative.csv       1038              144              50              50               244      cnv_2_conservative_1128genes          GSMB       0.20     0.495351
          03_standard.csv       1038              144              50              50               244          cnv_3_standard_1813genes          IAMB       0.20     0.498541
   04_fdr_significant.csv       1038              144              50              50               244   cnv_4_fdr_significant_1000genes          GSMB       0.10     0.504198
          05_balanced.csv       1038              144              50              50               244